In [3]:
import pandas as pd
import sqlite3

In [4]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [5]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [6]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

CERVEZAS
BARES
EMPLEADOS
REPARTO


In [ ]:
# crea las tablas

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS CERVEZAS (
  CodC INT PRIMARY KEY NOT NULL,
  Envase VARCHAR(10) NOT NULL,
  Capacidad DECIMAL(3,2) NOT NULL,
  Stock INT DEFAULT 0
);
'''

crsr.execute(query)

In [11]:
query = '''
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock) VALUES
(1, 'Botella', 0.2, 3600),
(2, 'Botella', 0.33, 1200),
(3, 'Lata', 0.33, 2400),
(4, 'Botella', 1, 288),
(5, 'Barril', 60, 30);
'''

crsr.execute(query)

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS BARES (
  CodB INT PRIMARY KEY NOT NULL,
  Nombre VARCHAR(10) NOT NULL,
  Cif VARCHAR(20),
  Localidad VARCHAR(20) NOT NULL
);
'''
crsr.execute(query)

In [10]:
query = '''
INSERT INTO BARES (CodB, Nombre, Cif, Localidad) VALUES
(1, 'Stop', '11111111X', 'Villa Botijo'),
(2, 'Las Vegas', '22222222Y', 'Villa Botijo'),
(3, 'Club Social', NULL, 'Las Ranas'),
(4, 'Otra Ronda', '33333333Z', 'La Esponja');
'''

crsr.execute(query)

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS EMPLEADOS (
  CodE INT PRIMARY KEY NOT NULL,
  Nombre VARCHAR(20) NOT NULL,
  Sueldo INT NOT NULL
);
'''
crsr.execute(query)

In [9]:
query = '''
INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo) VALUES
(1, 'Carlos Lopez', 120000),
(2, 'Rosa Perez', 110000),
(3, 'Luisa Garcia', 100000);
'''

crsr.execute(query)

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS REPARTO (
  CodE INT,
  CodB INT,
  CodC INT,
  Fecha VARCHAR(20),
  Cantidad INT,
  PRIMARY KEY (CodE, CodB, CodC, Fecha),
  FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
  FOREIGN KEY (CodB) REFERENCES BARES(CodB),
  FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
'''

crsr.execute(query)

In [8]:
query = '''
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad) VALUES
(1, 1, 1, '2005-10-21', 240),
(1, 1, 2, '2005-10-21', 48),
(1, 2, 3, '2005-10-22', 60),
(1, 4, 5, '2005-10-22', 4),
(2, 2, 3, '2005-10-22', 48),
(2, 2, 5, '2005-10-23', 2),
(2, 4, 1, '2005-10-23', 480),
(2, 4, 2, '2005-10-24', 72),
(3, 3, 3, '2005-10-24', 48),
(3, 3, 4, '2005-10-25', 20);
'''

crsr.execute(query)

In [ ]:
# Visualización de datos


In [12]:
query = '''
SELECT 
  e.Nombre AS Empleado,
  b.Nombre AS Bar,
  b.Localidad AS Localidad,
  c.Envase,
  c.Capacidad,
  r.Fecha,
  r.Cantidad
FROM REPARTO r
JOIN EMPLEADOS e ON r.CodE = e.CodE
JOIN BARES b ON r.CodB = b.CodB
JOIN CERVEZAS c ON r.CodC = c.CodC
'''

sql_query(query)

,Empleado,Bar,Localidad,Envase,Capacidad,Fecha,Cantidad
0,Carlos Lopez,Stop,Villa Botijo,Botella,0.20,2005-10-21,240
1,Carlos Lopez,Stop,Villa Botijo,Botella,0.33,2005-10-21,48
2,Carlos Lopez,Las Vegas,Villa Botijo,Lata,0.33,2005-10-22,60
3,Carlos Lopez,Otra Ronda,La Esponja,Barril,60.00,2005-10-22,4
4,Rosa Perez,Las Vegas,Villa Botijo,Lata,0.33,2005-10-22,48
5,Rosa Perez,Las Vegas,Villa Botijo,Barril,60.00,2005-10-23,2
6,Rosa Perez,Otra Ronda,La Esponja,Botella,0.20,2005-10-23,480
7,Rosa Perez,Otra Ronda,La Esponja,Botella,0.33,2005-10-24,72
8,Luisa Garcia,Club Social,Las Ranas,Lata,0.33,2005-10-24,48
9,Luisa Garcia,Club Social,Las Ranas,Botella,1.00,2005-10-25,20


In [11]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = '''
SELECT DISTINCT e.Nombre
FROM REPARTO r
JOIN EMPLEADOS e ON r.CodE = e.CodE
JOIN BARES b ON r.CodB = b.CodB
WHERE b.Nombre = 'Stop'
AND r.Fecha BETWEEN '2005-10-17' AND '2005-10-23'
'''


sql_query(query)

,Nombre
0,Carlos Lopez


In [12]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''
SELECT DISTINCT b.Cif, b.Nombre
FROM REPARTO r
JOIN BARES b ON r.CodB = b.CodB
JOIN CERVEZAS c ON r.CodC = c.CodC
WHERE c.Envase = 'Botella'
AND c.Capacidad < 1
ORDER BY b.Localidad
'''

sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [13]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''
SELECT 
  b.Nombre AS Bar,
  c.Envase,
  c.Capacidad,
  r.Fecha,
  r.Cantidad
FROM REPARTO r
JOIN BARES b ON r.CodB = b.CodB
JOIN CERVEZAS c ON r.CodC = c.CodC
'''

sql_query(query)

,Bar,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4
4,Las Vegas,Lata,0.33,2005-10-22,48
5,Las Vegas,Barril,60.00,2005-10-23,2
6,Otra Ronda,Botella,0.20,2005-10-23,480
7,Otra Ronda,Botella,0.33,2005-10-24,72
8,Club Social,Lata,0.33,2005-10-24,48
9,Club Social,Botella,1.00,2005-10-25,20


In [ ]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''

'''
sql_query(query)

In [ ]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''

'''
sql_query(query)

In [ ]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''

'''
sql_query(query)

In [ ]:
query = '''

'''
sql_query(query)

In [ ]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''

'''
sql_query(query)

In [ ]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''

'''
sql_query(query)

In [ ]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''

'''
sql_query(query)

In [ ]:
query = '''

'''
sql_query(query)

In [ ]:
query = '''
   
'''
sql_query(query)

In [ ]:
query = '''

'''
crsr.execute(query)

In [ ]:
query = '''

'''
sql_query(query)